# A Gallery of Common Distributions

| Difficulty | ███░░░░░░░ 3/10 |
| :--- | :--- |
| Prerequisites | 03.05 |
| Time | ~60 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/03_Probability_and_Statistics/06_common_distributions_gallery.ipynb)

---

## 🎯 Learning Objective

Tour the distributions that appear constantly in ML — Gaussian, Bernoulli, Categorical, Poisson, Exponential, Beta, Dirichlet, Gamma — and learn when each one is the natural choice.

---

## 📐 Theory

You now have the full grammar of probability (`03.01`-`03.05`). This notebook is a field guide
to the specific distributions that recur constantly in ML — what each one models, its
parameters, and its moments in terms of those parameters.

### Bernoulli and Categorical: the building blocks of classification

**Bernoulli($p$)** is a single coin flip: $X\in\{0,1\}$, $P(X=1)=p$, $\mathbb{E}[X]=p$,
$\text{Var}(X)=p(1-p)$. **Categorical($p_1,\ldots,p_K$)** generalizes it to $K$ outcomes each
with its own probability, $\sum_k p_k=1$ — exactly the distribution a softmax layer produces
over classes or vocabulary tokens.

### Binomial and Poisson: counting things

**Binomial($n,p$)** counts successes in $n$ independent Bernoulli($p$) trials: $P(X=k) =
\binom{n}{k}p^k(1-p)^{n-k}$, $\mathbb{E}[X]=np$, $\text{Var}(X)=np(1-p)$ — the sum of $n$ iid
Bernoullis, by the linearity of expectation from `03.03`. **Poisson($\lambda$)** counts events
in a fixed interval when events occur independently at a constant average rate $\lambda$:
$P(X=k) = \lambda^k e^{-\lambda}/k!$, with the unusual property $\mathbb{E}[X]=\text{Var}(X)=
\lambda$. Poisson is literally the limit of Binomial($n,p$) as $n\to\infty$, $p\to0$, with
$np=\lambda$ fixed — many rare independent chances collapsing into one rate.

### Gaussian: the default continuous distribution

**Gaussian/Normal($\mu,\sigma^2$)**: $f(x) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$,
$\mathbb{E}[X]=\mu$, $\text{Var}(X)=\sigma^2$. Its ubiquity isn't arbitrary: the **Central Limit
Theorem** says the sum of many independent, identically distributed random variables
approaches a Gaussian regardless of their own distribution — noise from many small independent
sources tends to look Gaussian, which is exactly why it's the default noise/prior assumption
across ML.

### Exponential and Gamma: waiting times

**Exponential($\lambda$)** models the waiting time until a Poisson-rate event occurs:
$f(x)=\lambda e^{-\lambda x}$ for $x\ge0$, $\mathbb{E}[X]=1/\lambda$. It's the unique continuous
distribution with the **memoryless property** $P(X>s+t\mid X>s)=P(X>t)$ — how long you've
already waited tells you nothing about how much longer you'll wait. **Gamma($k,\lambda$)**
generalizes it to the waiting time until the $k$-th event, and is the conjugate prior for a
Poisson rate or for the *precision* (inverse variance) of a Gaussian.

### Beta and Dirichlet: distributions over probabilities

**Beta($\alpha,\beta$)** is a distribution over $[0,1]$ — perfect for representing uncertainty
about a probability itself (a coin's bias, in `03.05`'s example), with $\mathbb{E}[X] =
\alpha/(\alpha+\beta)$. It is the conjugate prior to the Bernoulli/Binomial likelihood.
**Dirichlet($\alpha_1,\ldots,\alpha_K$)** is Beta's multivariate generalization: a distribution
over the probability simplex $\{(p_1,\ldots,p_K): p_k\ge0,\sum_k p_k=1\}$ — a distribution over
*entire categorical distributions*. It is conjugate to the Categorical likelihood, making it the
natural prior whenever a model's parameters are themselves a probability vector (topic
proportions in topic models, class priors, attention-weight priors).

### Choosing a distribution: matching support to structure

The right distribution is the one whose **support** (the set of values it can take) and
built-in assumptions match your data or parameter's actual constraints: binary outcomes →
Bernoulli; one-of-$K$ choices → Categorical; non-negative counts → Poisson; non-negative
continuous, possibly skewed → Exponential/Gamma; a bounded probability → Beta; a vector that
must sum to 1 → Dirichlet; everything else, by default and by the Central Limit Theorem →
Gaussian. The table built below turns this into a concrete lookup.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import gammaln
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

A gallery of the distributions above, each plotted with a few different parameter settings, so
their shapes and how their parameters control them are visible side by side.

In [ ]:
# A gallery: each distribution plotted at a few parameter settings, using the SAME
# .pdf()/.pmf() pattern established in 03.02 -- the API is identical no matter the distribution.
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

k_vals = np.arange(0, 11)
for n, p in [(10, 0.3), (10, 0.5), (10, 0.7)]:
    axes[0, 0].plot(k_vals, stats.binom(n, p).pmf(k_vals), "o-", label=f"n={n}, p={p}", ms=4)
axes[0, 0].set_title("Binomial(n, p)"); axes[0, 0].legend(fontsize=8); axes[0, 0].set_xlabel("k")

for lam in [1, 4, 9]:
    axes[0, 1].plot(k_vals, stats.poisson(lam).pmf(k_vals), "o-", label=f"lambda={lam}", ms=4)
axes[0, 1].set_title("Poisson(lambda)"); axes[0, 1].legend(fontsize=8); axes[0, 1].set_xlabel("k")

x_gauss = np.linspace(-6, 6, 300)
for mu, sigma in [(0, 1), (0, 2), (-2, 0.7)]:
    axes[0, 2].plot(x_gauss, stats.norm(mu, sigma).pdf(x_gauss), label=f"mu={mu}, sigma={sigma}")
axes[0, 2].set_title("Gaussian(mu, sigma)"); axes[0, 2].legend(fontsize=8); axes[0, 2].set_xlabel("x")

x_pos = np.linspace(0, 5, 300)
for lam in [0.5, 1, 2]:
    axes[1, 0].plot(x_pos, stats.expon(scale=1 / lam).pdf(x_pos), label=f"lambda={lam}")
axes[1, 0].set_title("Exponential(lambda)"); axes[1, 0].legend(fontsize=8); axes[1, 0].set_xlabel("x")

for k_shape, theta_scale in [(1, 1), (2, 1), (5, 1)]:
    axes[1, 1].plot(x_pos, stats.gamma(k_shape, scale=theta_scale).pdf(x_pos), label=f"k={k_shape}, theta={theta_scale}")
axes[1, 1].set_title("Gamma(k, theta) [k=1 -> Exponential]"); axes[1, 1].legend(fontsize=8); axes[1, 1].set_xlabel("x")

x_unit = np.linspace(0.001, 0.999, 300)
for a, b in [(0.5, 0.5), (2, 2), (2, 5)]:
    axes[1, 2].plot(x_unit, stats.beta(a, b).pdf(x_unit), label=f"alpha={a}, beta={b}")
axes[1, 2].set_title("Beta(alpha, beta) -- lives on [0,1]"); axes[1, 2].legend(fontsize=8); axes[1, 2].set_xlabel("x")

plt.tight_layout()
plt.show()

print("Binomial(10,0.5) mean:", stats.binom(10, 0.5).mean(), " (expect n*p=5.0)")
print("Poisson(4) mean == variance:", stats.poisson(4).mean(), "==", stats.poisson(4).var(), " (defining property)")
print("Gamma(1, scale=2) matches Exponential(scale=2)? ",
      np.allclose(stats.gamma(1, scale=2).pdf(x_pos), stats.expon(scale=2).pdf(x_pos)))

## 🐍 Implementation from Scratch

We implement the Categorical and Dirichlet distributions from scratch (sampling and density),
verify the Binomial-as-limit-of-Poisson connection numerically, and cross-check everything
against `scipy.stats`.

In [ ]:
rng = np.random.default_rng(0)

# Categorical sampling from scratch: inverse-transform sampling (03.02) applied to a PMF over
# a finite unordered set of outcomes, via the CDF of the category INDEX.
def categorical_sample_from_scratch(probs, size, rng):
    cdf = np.cumsum(probs)
    u = rng.uniform(0, 1, size=size)
    return np.searchsorted(cdf, u)  # first index where cdf exceeds u -- inverse transform

category_probs = np.array([0.1, 0.2, 0.4, 0.3])
samples = categorical_sample_from_scratch(category_probs, size=50_000, rng=rng)
empirical_freqs = np.bincount(samples, minlength=4) / len(samples)
print("Categorical from-scratch empirical frequencies:", np.round(empirical_freqs, 3))
print("True probabilities:                             ", category_probs)

# Dirichlet PDF from scratch, straight from its definition, checked against scipy.stats.dirichlet
def dirichlet_pdf_from_scratch(x, alpha):
    # f(x) = [1/B(alpha)] * prod_k x_k^(alpha_k - 1), B(alpha) = prod Gamma(alpha_k) / Gamma(sum alpha_k)
    log_beta_norm = np.sum(gammaln(alpha)) - gammaln(np.sum(alpha))
    log_pdf = np.sum((alpha - 1) * np.log(x)) - log_beta_norm
    return np.exp(log_pdf)

alpha_vec = np.array([2.0, 3.0, 4.0])
test_point = np.array([0.2, 0.3, 0.5])
ours = dirichlet_pdf_from_scratch(test_point, alpha_vec)
scipy_answer = stats.dirichlet(alpha_vec).pdf(test_point)
print(f"\nDirichlet PDF at {test_point}: ours={ours:.6f}, scipy={scipy_answer:.6f}, "
      f"diff={abs(ours - scipy_answer):.2e}")

# A Dirichlet sample IS itself a valid categorical distribution -- sample one, check it sums to 1
one_dirichlet_draw = stats.dirichlet(alpha_vec).rvs(random_state=0)[0]
print(f"One Dirichlet({alpha_vec}) draw: {np.round(one_dirichlet_draw, 4)} (sums to {one_dirichlet_draw.sum():.6f})")

# Binomial -> Poisson limit: fix lambda = n*p, shrink p while growing n, and watch the
# Binomial PMF converge pointwise to the Poisson(lambda) PMF
lam_fixed = 3.0
k_vals = np.arange(0, 10)
poisson_pmf = stats.poisson(lam_fixed).pmf(k_vals)
print(f"\nPoisson(lambda={lam_fixed}) PMF:              {np.round(poisson_pmf, 5)}")
for n in [30, 300, 3000]:
    p = lam_fixed / n
    binom_pmf = stats.binom(n, p).pmf(k_vals)
    max_diff = np.max(np.abs(binom_pmf - poisson_pmf))
    print(f"Binomial(n={n:4d}, p={p:.5f}) PMF:  max diff from Poisson = {max_diff:.6f}")
print("(max difference shrinks as n grows -- confirms Poisson is the n->inf, p->0, np=lambda limit)")

## 🤖 Why This Matters for AI

Every classification model's output layer is a **Categorical** distribution (softmax); every
topic model or Bayesian treatment of class proportions reaches for a **Dirichlet** prior because
it's conjugate to Categorical, turning posterior updates into simple parameter addition rather
than an intractable integral. Below, we run that update explicitly: start from a Dirichlet prior
over a language model's next-token distribution, observe token counts, and get a closed-form
Dirichlet posterior — the categorical/Dirichlet analogue of `03.05`'s Beta-Bernoulli example.

---

In [ ]:
# Dirichlet-Categorical conjugacy: a Bayesian view of "estimating a language model's next-token
# distribution" -- start with a prior belief, observe token counts, get a closed-form posterior.
vocab = ["the", "a", "cat", "dog", "runs"]
V = len(vocab)

# A weak, symmetric Dirichlet prior: mild belief that all tokens are roughly equally likely
prior_alpha = np.ones(V) * 2.0

# Observed counts of each token in a small corpus sample
observed_counts = np.array([40, 25, 15, 12, 8])
print(f"Vocabulary: {vocab}")
print(f"Observed counts: {observed_counts}  (total {observed_counts.sum()} tokens seen)")

# Dirichlet-Categorical conjugacy: posterior alpha = prior alpha + observed counts. Closed form,
# exactly like the Beta-Bernoulli update in 03.05, just generalized from 2 categories to V.
posterior_alpha = prior_alpha + observed_counts
posterior_mean_dist = posterior_alpha / posterior_alpha.sum()  # E[p_k] for a Dirichlet(alpha) is alpha_k/sum(alpha)

# Compare to the plain MLE estimate (normalized counts, no prior at all -- 03.07's approach)
mle_dist = observed_counts / observed_counts.sum()

print("\nToken   |  MLE estimate  |  Bayesian posterior mean")
for tok, mle_p, post_p in zip(vocab, mle_dist, posterior_mean_dist):
    print(f"{tok:7s} |     {mle_p:.4f}     |          {post_p:.4f}")

# The prior matters most when data is SCARCE -- rerun with far fewer observed tokens
sparse_counts = np.array([2, 1, 0, 0, 0])   # "dog" and "runs" never observed yet
posterior_alpha_sparse = prior_alpha + sparse_counts
posterior_mean_sparse = posterior_alpha_sparse / posterior_alpha_sparse.sum()
mle_sparse = sparse_counts / sparse_counts.sum()

print(f"\nWith only {sparse_counts.sum()} tokens observed (two tokens never seen):")
print(f"  MLE estimate           : {np.round(mle_sparse, 4)}  <- assigns EXACTLY ZERO probability to unseen tokens")
print(f"  Bayesian posterior mean: {np.round(posterior_mean_sparse, 4)}  <- prior keeps unseen tokens' probability > 0")
print("This is exactly why Bayesian/Dirichlet smoothing (a direct ancestor of Laplace/add-k")
print("smoothing) matters for language models: it avoids assigning zero probability -- and")
print("hence infinite surprise/cross-entropy -- to a token never observed during training.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Using `scipy.stats`, compute the mean and variance of `Gamma(k=3, theta=2)` two ways: (a) via
   `.mean()`/`.var()`, and (b) via the formulas $\mathbb{E}[X]=k\theta$,
   $\text{Var}(X)=k\theta^2$. As a sanity check, confirm the variance is non-negative (guaranteed
   for any $k,\theta>0$, but worth checking your arithmetic against), and confirm both methods
   match.

<details>
<summary>💡 Solution</summary>

By the formulas, $\mathbb{E}[X] = k\theta = 3\times 2 = 6$ and $\text{Var}(X) = k\theta^2 =
3\times 4 = 12$ — non-negative, as required. `scipy.stats.gamma(3, scale=2).mean()` returns
`6.0` and `.var()` returns `12.0`, matching the formula-based computation exactly.

</details>

### 💭 UNDERSTAND
2. Poisson($\lambda$) is the limit of Binomial($n,p$) as $n\to\infty,p\to0$ with $np=\lambda$
   fixed. Two binomials share the same $\lambda=np=3$: Binomial($n{=}1000,p{=}0.003$) and
   Binomial($n{=}20,p{=}0.15$). Without computing anything, predict which one's PMF is closer to
   Poisson(3)'s PMF, and explain why matching $np$ alone is not sufficient for a good
   approximation — then verify numerically.

<details>
<summary>💡 Solution</summary>

Binomial($n{=}1000,p{=}0.003$) should be much closer to Poisson(3), because the Poisson limit
requires $n\to\infty$ *and* $p\to0$ together, not merely a fixed product $np$ — a large $n$ with
correspondingly tiny $p$ approximates "many independent chances, each individually rare" far
better than a small $n$ with a not-so-tiny $p$ (here $p=0.15$ is not close to $0$ at all).
Numerically, the maximum pointwise difference between the Binomial and Poisson(3) PMFs (over
$k=0,\ldots,9$) is only about $0.0003$ for $n{=}1000,p{=}0.003$, versus about $0.019$ — roughly
$60\times$ larger — for $n{=}20,p{=}0.15$, confirming that matching $\lambda=np$ alone is not
enough; $n$ itself needs to be large (and $p$ correspondingly small) for the approximation to be
tight.

</details>

### ✏️ DERIVE
3. Derive the Poisson PMF as the limit of the Binomial PMF, starting from
   $P(X=k) = \binom{n}{k}p^k(1-p)^{n-k}$ with $p=\lambda/n$, and taking $n\to\infty$ with
   $k,\lambda$ fixed. (Hint: expand $\binom{n}{k} = \frac{n!}{k!(n-k)!}$, group the $n$-dependent
   factors, and recall $\lim_{n\to\infty}(1-\lambda/n)^n = e^{-\lambda}$.)

<details>
<summary>💡 Solution outline</summary>

Substitute $p=\lambda/n$: $\binom{n}{k}\left(\frac{\lambda}{n}\right)^k\left(1-\frac{\lambda}{n}\right)^{n-k}
= \frac{n!}{k!(n-k)!}\cdot\frac{\lambda^k}{n^k}\cdot\left(1-\frac{\lambda}{n}\right)^{n-k}$.
Rewrite $\frac{n!}{(n-k)!} = n(n-1)\cdots(n-k+1)$, a product of $k$ terms each $\to n$ as
$n\to\infty$, so $\frac{n!}{(n-k)!\,n^k}\to 1$. Split
$\left(1-\frac{\lambda}{n}\right)^{n-k} = \left(1-\frac{\lambda}{n}\right)^n
\left(1-\frac{\lambda}{n}\right)^{-k}$; the second factor $\to 1^{-k}=1$ as $n\to\infty$ (fixed
$k$), and the first factor $\to e^{-\lambda}$ by the standard limit. Combining all three limits:
$$\lim_{n\to\infty}\binom{n}{k}\left(\frac{\lambda}{n}\right)^k\left(1-\frac{\lambda}{n}\right)^{n-k}
= \frac{\lambda^k}{k!}\cdot 1\cdot e^{-\lambda} = \frac{\lambda^k e^{-\lambda}}{k!},$$
exactly the Poisson($\lambda$) PMF — confirming the notebook's own numerical demonstration
(Binomial PMF converging to Poisson's as $n$ grows with $\lambda=np$ fixed) algebraically.

</details>

### 🐍 IMPLEMENT
4. Verify the Beta-Binomial conjugacy relationship used in `03.05`: for a Beta($\alpha,\beta$)
   prior updated with $k$ successes out of $n$ Bernoulli trials, the posterior is
   Beta($\alpha+k,\beta+n-k$). Pick concrete numbers (e.g. $\alpha=2,\beta=5,k=7,n=10$), compute
   the posterior mean in closed form using $\mathbb{E}[\text{Beta}(a,b)]=a/(a+b)$, and check it
   against `scipy.stats.beta(alpha+k, beta+n-k).mean()`.

<details>
<summary>✅ How to know you're right</summary>

With $\alpha=2,\beta=5,k=7,n=10$: posterior parameters are $a=2+7=9$, $b=5+3=8$, so the
closed-form posterior mean is $9/17\approx 0.5294$. `scipy.stats.beta(9, 8).mean()` should
return the identical value (`0.5294...`) to full floating-point precision — since both
computations are evaluating the exact same formula, any mismatch beyond rounding indicates an
arithmetic slip in the by-hand computation of $a$ or $b$, not a modeling discrepancy.

</details>

### 🤖 APPLY
5. In the Dirichlet-Categorical smoothing cell, sweep the prior strength by scaling
   `prior_alpha` by factors `[0.1, 1, 10, 100]` (keeping it symmetric across all 5 tokens) while
   holding `sparse_counts` fixed. Record the posterior mean probability assigned to the
   never-observed token `"dog"` at each factor.

<details>
<summary>✅ What you should observe</summary>

The posterior probability assigned to `"dog"` should increase monotonically with the scaling
factor: roughly $0.05\to0.15\to0.19\to0.20$ as the factor goes $0.1\to1\to10\to100$ in a
representative run, asymptotically approaching $1/V=0.2$ (the fully uniform prior's own
per-token probability) as the prior comes to dominate the small amount of observed data. At the
weakest prior setting (factor $0.1$), the posterior stays close to the sparse MLE estimate
(which would assign exactly $0$ to `"dog"`), while at the strongest setting it is nearly
indistinguishable from the symmetric prior itself, having been barely updated by only 3 observed
tokens.

</details>

### 🚀 CHALLENGE
6. Section "Why This Matters for AI" argues Dirichlet/Bayesian smoothing avoids assigning exactly
   zero probability to unseen tokens, unlike plain MLE. Investigate the tradeoff exercise 5
   raises directly: using the notebook's `observed_counts` (NOT `sparse_counts` — a much larger,
   less sparse sample), sweep the same prior-strength factors `[0.1, 1, 10, 100]` and compute, at
   each factor, some measure of how far the posterior mean distribution has drifted from the
   plain MLE distribution (e.g. total variation distance $\frac12\sum_k|p_k^{\text{post}} -
   p_k^{\text{MLE}}|$, or just eyeball the printed table). Contrast this drift with the same
   sweep applied to `sparse_counts`. Explain why a language model builder would NOT simply pick
   the largest prior strength that maximizes robustness to unseen tokens, connecting your answer
   to the bias-variance tradeoff from `03.05`'s regularization discussion.

<details>
<summary>🔍 What a strong answer covers</summary>

With the larger `observed_counts` sample (75 tokens total), even a fairly strong prior (factor
$10$, i.e. $\alpha_i=20$ each, total prior mass $100$ — comparable to or larger than the data)
still lets the posterior mean track the MLE distribution reasonably well, since the *ratio* of
data evidence to prior mass is what matters, not the data's absolute count alone. With
`sparse_counts` (only 3 tokens observed), the exact same prior strengths distort the posterior
distribution much more severely, as exercise 5 already showed — because there the prior mass
heavily outweighs the tiny amount of real evidence. A strong answer computes the total variation
distance (or equivalent) explicitly at each factor for both count arrays and shows the
sparse-data curve moves much further from the MLE at the same prior strength than the
richer-data curve does. The conclusion strong answers reach: a prior strong enough to handle
worst-case data scarcity gracefully will, on more plentifully-observed tokens, pull the estimate
away from what the data actually supports — exactly the same bias-(robustness)-variance tension
`03.05` describes for ridge regression's $\lambda$, just now controlling robustness to *unseen*
categories rather than parameter magnitude. The practical resolution (as in real smoothing
schemes like Kneser-Ney) is typically a smaller, carefully tuned prior strength, or a prior
whose strength doesn't stay fixed but decreases relative to how much data has been observed.

</details>

---

## 📚 Further Reading
- Murphy, *Probabilistic Machine Learning: An Introduction*, Ch. 2.2–2.3 (Common probability distributions)
- Bishop, *Pattern Recognition and Machine Learning*, Ch. 2 (Probability Distributions)
- `scipy.stats` [continuous distributions reference](https://docs.scipy.org/doc/scipy/reference/stats.html#continuous-distributions)

---

*Next notebook: [Maximum Likelihood Estimation](07_maximum_likelihood_estimation.ipynb)*